# Unidad 5 — Sistemas Multi-Agente Modernos
## Notebook 03: CrewAI — Sistemas Multi-Agente con Roles `[Intermedio ★★★☆☆]`

**Duración estimada:** 5-6 horas  
**Entorno:** `ia_multiagente` (Python 3.11)  
**Modelo por defecto:** Gemini 2.0 Flash · Alternativa: Ollama Llama 3.2  
**Costo estimado:** ~$0.03 con Gemini Flash · $0 con Ollama  
**Skill que se crea:** `external_skills/evaluation/output_scorer.py`

---

## Tabla de Contenidos

1. [Warm-Up y Repaso de U5_02](#1-warmup)
2. [¿Por qué CrewAI? El modelo de tripulación](#2-por-que-crewai)
3. [Primitivas de CrewAI: Agent, Task, Crew, Process](#3-primitivas)
   - 3.1 Agent — Role, Goal, Backstory
   - 3.2 Task — description, expected_output
   - 3.3 Crew y Process: Sequential vs Hierarchical
4. [Tools en CrewAI](#4-tools)
   - 4.1 Tools nativas de CrewAI
   - 4.2 Bridging: tools de LangChain en CrewAI
5. [Process Hierarchical: Manager + Delegates](#5-hierarchical)
6. [Skill: Output Scorer — Evaluación Automática](#6-output-scorer)
7. [Proyecto: Equipo de Investigación en Nanotecnología](#7-proyecto)
   - 7.1 Definir el equipo
   - 7.2 Ejecutar la misión
   - 7.3 Evaluar outputs con la skill
8. [Resumen y Criterios de Evaluación](#8-resumen)

---

## Objetivos de Aprendizaje

Al completar esta notebook serás capaz de:
- Definir `Agent`, `Task` y `Crew` con configuración completa de roles y backstories
- Comparar el `Process.sequential` vs `Process.hierarchical` con un ejemplo ejecutado
- Reutilizar tools de LangChain dentro de agentes CrewAI
- Evaluar la calidad de un output multi-agente usando la skill `output_scorer`
- Construir un equipo de investigación de 4 agentes para una tarea de nanotecnología

**Prerequisito:** U5_01 (tools, skills), U5_02 (LangGraph, routing)

---
## 1. Warm-Up y Repaso de U5_02 <a id='1-warmup'></a>

In [1]:
# ============================================================
# WARM-UP: Entorno y Dependencias — U5_03
# ============================================================
import subprocess, sys, os
from pathlib import Path

def check_install(package, import_name=None):
    name = import_name or package.split('==')[0].replace('-', '_')
    try:
        __import__(name)
        print(f"  OK: {package}")
    except ImportError:
        print(f"  Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"  Instalado: {package}")

packages = [
    ("crewai==0.80.0",                   "crewai"),
    ("crewai-tools==0.14.0",             "crewai_tools"),
    ("langchain-openai==0.3.12",         "langchain_openai"),
    ("langchain-google-genai==2.1.4",    "langchain_google_genai"),
    ("python-dotenv==1.1.0",             "dotenv"),
]

print("Verificando paquetes...")
for pkg, imp in packages:
    check_install(pkg, imp)

# Fijar openai a versión compatible con langchain_openai 0.3.12
# openai >= 1.79 introdujo resources/videos.py con un bug que rompe ChatOpenAI
import openai as _oai
_ver = tuple(int(x) for x in _oai.__version__.split(".")[:2])
if _ver >= (1, 79):
    print(f"  [!] openai {_oai.__version__} incompatible con langchain_openai 0.3.12")
    print("      Reinstalando versión compatible (esto puede tardar 10-20s)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "openai>=1.58.0,<1.79.0", "--force-reinstall", "-q"
    ])
    print("  openai reinstalado correctamente.")
    print()
    print("  *** IMPORTANTE: REINICIA el kernel ahora ***")
    print("  Menú → Kernel → Restart Kernel, luego vuelve a ejecutar desde la celda 1")
    raise SystemExit("Reinicia el kernel: Kernel > Restart Kernel and Run All Cells")
else:
    print(f"  OK: openai {_oai.__version__}")

from dotenv import load_dotenv
load_dotenv(override=True)

# project root
project_root = Path.cwd()
for _ in range(5):
    if (project_root / "external_skills").exists():
        break
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Detectar backends disponibles
google_key     = os.environ.get("GOOGLE_API_KEY")
openrouter_key = os.environ.get("OPENROUTER_API_KEY")

backend = (
    "OpenRouter"     if openrouter_key else
    "Gemini"         if google_key     else
    "Sin backend — configura OPENROUTER_API_KEY o GOOGLE_API_KEY en .env"
)

print(f"\nProject root: {project_root}")
print(f"Backend LLM activo: {backend}")
print("Warm-up completado.")


Verificando paquetes...


  OK: crewai==0.80.0


  OK: crewai-tools==0.14.0


  OK: langchain-openai==0.3.12


  OK: langchain-google-genai==2.1.4
  OK: python-dotenv==1.1.0
  OK: openai 1.78.1

Project root: C:\IA Nanotecnología\Antigravity-Nano-Research-Multiagentic-Core-main
Backend LLM activo: OpenRouter
Warm-up completado.


In [2]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# Asegurar que el project_root (con external_skills) está en sys.path
_root = Path.cwd()
for _ in range(6):
    if (_root / "external_skills").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
    _root = _root.parent

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

OPENROUTER_HEADERS = {
    "HTTP-Referer": "https://github.com/Antigravity-Nano-Research",
    "X-Title": "U5_03 CrewAI Nanotecnologia",
}

if OPENROUTER_API_KEY:
    # Lista de modelos de pago (más económicos primero)
    OPENROUTER_MODELS = [
        "google/gemini-2.5-flash",
        "anthropic/claude-3-haiku",
        "openai/gpt-4o-mini",
        "meta-llama/llama-3.1-8b-instruct",
    ]

    llm = None
    for model_id in OPENROUTER_MODELS:
        try:
            _candidate = ChatOpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=OPENROUTER_API_KEY,
                model=model_id,
                temperature=0.2,
                default_headers=OPENROUTER_HEADERS,
            )
            # Verificar que funciona con una llamada simple
            _test = _candidate.invoke("OK")
            if _test:
                llm = _candidate
                print(f"LLM: OpenRouter — {model_id}")
                break
        except Exception as e:
            msg = str(e).lower()
            if any(x in msg for x in ("429", "404", "rate", "quota")):
                print(f"  [!] {model_id} no disponible, probando siguiente...")
                continue
            raise

    if llm is None:
        raise RuntimeError(
            "Todos los modelos OpenRouter no disponibles. "
            "Verifica tu OPENROUTER_API_KEY y créditos."
        )

elif GOOGLE_API_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.2,
    )
    print("LLM: Gemini 2.0 Flash (Google AI)")

else:
    raise EnvironmentError("Configura OPENROUTER_API_KEY o GOOGLE_API_KEY en .env")

# Repaso U5_02: verificar skills disponibles
from external_skills.routing.task_classifier import classify


LLM: OpenRouter — google/gemini-2.5-flash


---
## 2. ¿Por qué CrewAI? El Modelo de Tripulación <a id='2-por-que-crewai'></a>

LangGraph es excelente para flujos de trabajo complejos y control fino del estado. Pero hay casos de uso donde pensamos **naturalmente en roles humanos**: un equipo de investigación tiene un investigador senior, un analista de datos, un redactor técnico y un revisor de calidad. CrewAI mapea directamente a esa intuición.

**CrewAI vs LangGraph — cuándo usar cada uno:**

| Criterio | CrewAI | LangGraph |
|---------|--------|----------|
| **Paradigma mental** | Equipo de personas con roles | Máquina de estados / flujo de proceso |
| **Fortaleza** | Colaboración natural entre agentes con personalidades | Control fino del flujo, ciclos complejos, state management |
| **Curva de aprendizaje** | Más baja — intuitivo para no-ingenieros | Más alta — requiere pensar en grafos |
| **Delegación** | Process.hierarchical: Manager asigna dinamicamente | Manual: nodos explícitos en el grafo |
| **Observabilidad** | CrewAI+ dashboard (cloud) | LangSmith (tracing) |
| **Personalización** | Media | Alta |
| **Producción enterprise** | CrewAI Enterprise / CrewAI+ | LangGraph Platform |

La regla práctica: **si tu problema se describe naturalmente como "un equipo donde X hace esto y Y hace aquello", usa CrewAI. Si necesitas control de flujo complejo con ciclos y condiciones intrincadas, usa LangGraph.**

Y se pueden combinar: un LangGraph puede tener nodos que internamente son Crews de CrewAI.

---
## 3. Primitivas de CrewAI: Agent, Task, Crew, Process <a id='3-primitivas'></a>

### 3.1 Agent — Role, Goal, Backstory

Un `Agent` en CrewAI tiene tres campos fundamentales que determinan su comportamiento:

- **role:** Título del cargo — lo que ES el agente. Ej: "Investigador Senior en IA"
- **goal:** Objetivo principal que guía sus decisiones. Ej: "Encontrar papers relevantes y sintetizar el estado del arte"
- **backstory:** Historia de fondo que establece expertise, estilo y perspectiva. Esto va al system prompt y tiene un impacto dramático en la calidad de las respuestas

La **backstory** es donde se inyecta el equivalente al warm-up de skill de U5_01. Cuanto más específica y detallada es la backstory, mejor desempeña el agente su rol.

In [3]:
# ============================================================
# WARM-UP: Sección 3 — Imports de CrewAI
# ============================================================
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

print("CrewAI imports OK")

CrewAI imports OK


In [4]:
# ────────────────────────────────────────────────────────────
# Output esperado: dos agentes con roles y backstories distintas
# ────────────────────────────────────────────────────────────

import os
import logging
from dotenv import load_dotenv
from crewai import Agent, LLM

logging.getLogger("litellm").setLevel(logging.ERROR)

load_dotenv(override=True)

openrouter_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_key:
    raise ValueError("OPENROUTER_API_KEY no encontrada en .env")

# Lista de modelos por rol (más económicos primero)
MODELS_INVESTIGADOR = [
    "google/gemini-2.5-flash",
    "anthropic/claude-3-haiku",
    "openai/gpt-4o-mini",
]

MODELS_REDACTOR = [
    "meta-llama/llama-3.1-8b-instruct",
    "google/gemini-2.5-flash",
    "anthropic/claude-3-haiku",
]


def crear_llm_con_fallback(models_list, temperature, max_tokens, rol_nombre):
    """Crea un LLM nativo de CrewAI via OpenRouter con fallback automático entre modelos.
    Usa el prefijo 'openrouter/' para que litellm enrute a OpenRouter
    en lugar del proveedor nativo de CrewAI (evita 403 con GOOGLE_API_KEY)."""
    for model_id in models_list:
        try:
            _candidate = LLM(
                model=f"openrouter/{model_id}",
                api_key=openrouter_key,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            # Verificar que funciona con una llamada simple
            _test = _candidate.call([{"role": "user", "content": "OK"}])
            if _test:
                print(f"  {rol_nombre}: {model_id}")
                return _candidate
        except Exception as e:
            msg = str(e).lower()
            if any(x in msg for x in ("429", "404", "rate", "quota", "permission", "denied")):
                print(f"  [!] {model_id} no disponible para {rol_nombre}, probando siguiente...")
                continue
            raise
    raise RuntimeError(f"Todos los modelos agotados para {rol_nombre}")


print("Configurando LLMs con fallback automático (via OpenRouter nativo):")

llm_investigador = crear_llm_con_fallback(
    MODELS_INVESTIGADOR, temperature=0.3, max_tokens=4000, rol_nombre="Investigador"
)
llm_redactor = crear_llm_con_fallback(
    MODELS_REDACTOR, temperature=0.7, max_tokens=3000, rol_nombre="Redactor"
)

print("\n✓ Ambos LLMs configurados correctamente\n")


# ============================================================
# VERIFICACIÓN DE MODELOS
# ============================================================
def verificar_modelo(llm_obj, nombre):
    try:
        response = llm_obj.call([{"role": "user", "content": "Responde solo con: OK"}])
        content = str(response)
        preview = (content[:60] + "...") if len(content) > 60 else content
        print(f"{nombre} - Modelo funcionando: {preview}")
        return True
    except Exception as e:
        print(f"{nombre} - Error: {e}")
        return False


print("=" * 60)
print("VERIFICANDO MODELOS OPENROUTER")
print("=" * 60)
verificar_modelo(llm_investigador, "Investigador")
verificar_modelo(llm_redactor, "Redactor")
print()

# ============================================================
# AGENTE 1: INVESTIGADOR
# ============================================================
investigador = Agent(
    role="Investigador Senior en IA y Nanotecnologia",
    goal=(
        "Recopilar, analizar y sintetizar literatura cientifica sobre el tema asignado. "
        "Identificar tendencias emergentes, gaps en el conocimiento y oportunidades de investigacion."
    ),
    backstory=(
        "Eres un investigador con 15 anos de experiencia en IA aplicada a nanomateriales. "
        "Has publicado en Nature Nanotechnology y ACS Nano. "
        "Tu metodologia combina revisiones sistematicas con sintesis cualitativa. "
        "Siempre citas fuentes especificas y distingues entre hallazgos replicados y prometedores pero preliminares. "
        "Tu estilo de escritura es preciso, denso en informacion y directo al punto."
        "\n\n"
        "IMPORTANTE: Basa tus respuestas en conocimiento cientifico establecido. "
        "Cuando menciones investigaciones, indica si son hallazgos replicados o resultados preliminares."
    ),
    llm=llm_investigador,
    verbose=True,
    max_iter=3,
    memory=True,
    allow_delegation=False,
)

# ============================================================
# AGENTE 2: REDACTOR
# ============================================================
redactor = Agent(
    role="Redactor Tecnico Senior",
    goal=(
        "Transformar informacion tecnica densa en documentos claros, bien estructurados y "
        "accesibles para audiencias con formacion universitaria en ciencias."
    ),
    backstory=(
        "Eres experto en comunicacion cientifica con experiencia en revistas de divulgacion como "
        "Scientific American y MIT Technology Review. "
        "Sabes convertir conceptos complejos en narrativas comprensibles sin perder rigor. "
        "Estructura tu output con headers, bullet points y ejemplos concretos. "
        "Siempre incluyes una seccion de 'Implicaciones Practicas' al final."
        "\n\n"
        "Estilo: Usa analogias accesibles, evita jerga innecesaria, "
        "y asegura que cada seccion fluya logicamente de una a la siguiente."
    ),
    llm=llm_redactor,
    verbose=True,
    max_iter=3,
    memory=True,
    allow_delegation=False,
)

print("=" * 60)
print("AGENTES CONFIGURADOS EXITOSAMENTE")
print("=" * 60)
print(f"\nInvestigador: {investigador.role}")
print(f"   Temperature: 0.3 (precision)")
print(f"   Max tokens: 4000")
print(f"\nRedactor: {redactor.role}")
print(f"   Temperature: 0.7 (creatividad)")
print(f"   Max tokens: 3000")
print("\n" + "=" * 60)
print("TODO LISTO - Los agentes estan preparados para trabajar")
print("=" * 60)


Configurando LLMs con fallback automático (via OpenRouter nativo):


  Investigador: google/gemini-2.5-flash


  Redactor: meta-llama/llama-3.1-8b-instruct

✓ Ambos LLMs configurados correctamente

VERIFICANDO MODELOS OPENROUTER


Investigador - Modelo funcionando: OK


Redactor - Modelo funcionando: OK



AGENTES CONFIGURADOS EXITOSAMENTE

Investigador: Investigador Senior en IA y Nanotecnologia
   Temperature: 0.3 (precision)
   Max tokens: 4000

Redactor: Redactor Tecnico Senior
   Temperature: 0.7 (creatividad)
   Max tokens: 3000

TODO LISTO - Los agentes estan preparados para trabajar



## Tabla de Modelos Gratuitos en OpenRouter

| Modelo | Código | Mejor Para | Temperature | Max Tokens | Contexto | Velocidad |
|--------|--------|------------|-------------|------------|----------|----------|
| **Gemini 2.0 Flash** | `google/gemini-2.0-flash-exp` | Investigación, análisis, razonamiento complejo | 0.3-0.5 | 8192 | 1M tokens | Muy rápida |
| **Gemini 2.0 Flash Lite** | `google/gemini-2.0-flash-lite-preview-02-05` | Procesamiento rápido, tareas simples | 0.5 | 8192 | 1M tokens | Extremadamente rápida |
| **Llama 3.2 3B** | `meta-llama/llama-3.2-3b-instruct` | Redacción, síntesis, creatividad | 0.7 | 8192 | 128K tokens | Rápida |
| **Llama 3.1 8B** | `meta-llama/llama-3.1-8b-instruct` | Equilibrio entre calidad y velocidad | 0.5-0.7 | 8192 | 128K tokens | Buena |
| **Qwen 2.5 7B** | `qwen/qwen-2.5-7b-instruct` | Análisis técnico, código, matemáticas | 0.3 | 8192 | 128K tokens | Buena |
| **Qwen 2.5 3B** | `qwen/qwen-2.5-3b-instruct` | Tareas ligeras, embeddings | 0.5 | 8192 | 32K tokens | Rápida |
| **Phi-3.5 Mini** | `microsoft/phi-3.5-mini-128k-instruct` | Texto claro, explicaciones educativas | 0.7 | 8192 | 128K tokens | Rápida |
| **Phi-3 Mini** | `microsoft/phi-3-mini-128k-instruct` | Documentación técnica, tutoriales | 0.6 | 8192 | 128K tokens | Rápida |
| **Mistral 7B** | `mistralai/mistral-7b-instruct` | Conversación, razonamiento general | 0.5 | 8192 | 32K tokens | Buena |
| **Mixtral 8x7B** | `mistralai/mixtral-8x7b-instruct` | Razonamiento complejo, multitarea | 0.4 | 8192 | 32K tokens | Media |
| **DeepSeek V2.5** | `deepseek/deepseek-v2.5` | Análisis profundo, código, matemáticas | 0.3 | 8192 | 128K tokens | Media |
| **DeepSeek Coder** | `deepseek/deepseek-coder-6.7b-instruct` | Programación, código técnico | 0.2 | 8192 | 16K tokens | Buena |
| **Hermes 3** | `nousresearch/hermes-3-llama-3.1-8b` | Creatividad, escritura literaria | 0.8 | 8192 | 128K tokens | Buena |
| **OpenChat 3.5** | `openchat/openchat-3.5-0106` | Conversación natural, chatbots | 0.6 | 8192 | 8K tokens | Rápida |
| **Zephyr 7B** | `huggingface/zephyr-7b-beta` | Asistencia general, seguimiento instrucciones | 0.5 | 8192 | 8K tokens | Rápida |

## Combinaciones Recomendadas

### Opción 1: Investigación + Redacción (Equilibrio)
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.3 |
| Redactor | Llama 3.2 3B | `meta-llama/llama-3.2-3b-instruct` | 0.7 |

### Opción 2: Máxima Calidad (Ambos con Gemini)
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.3 |
| Redactor | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.7 |

### Opción 3: Especializado Técnico
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Qwen 2.5 7B | `qwen/qwen-2.5-7b-instruct` | 0.3 |
| Redactor | Phi-3.5 Mini | `microsoft/phi-3.5-mini-128k-instruct` | 0.7 |

### Opción 4: Código y Matemáticas
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | DeepSeek V2.5 | `deepseek/deepseek-v2.5` | 0.2 |
| Redactor | Qwen 2.5 7B | `qwen/qwen-2.5-7b-instruct` | 0.5 |

### Opción 5: Creatividad y Narrativa
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Mistral 7B | `mistralai/mistral-7b-instruct` | 0.4 |
| Redactor | Hermes 3 | `nousresearch/hermes-3-llama-3.1-8b` | 0.8 |

### Opción 6: Rápido y Ligero
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash Lite | `google/gemini-2.0-flash-lite-preview-02-05` | 0.5 |
| Redactor | Qwen 2.5 3B | `qwen/qwen-2.5-3b-instruct` | 0.6 |

## Código de Ejemplo para Cambiar Combinaciones

```python
# Configuracion base para todos los modelos
def crear_llm(modelo, temperatura, max_tokens=4000):
    return ChatOpenAI(
        model=modelo,
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        temperature=temperatura,
        max_tokens=max_tokens,
        base_url="https://openrouter.ai/api/v1"
    )

# Opcion 1: Investigacion + Redaccion
llm_investigador = crear_llm("google/gemini-2.0-flash-exp", 0.3)
llm_redactor = crear_llm("meta-llama/llama-3.2-3b-instruct", 0.7)

# Opcion 2: Ambos Gemini
llm_investigador = crear_llm("google/gemini-2.0-flash-exp", 0.3)
llm_redactor = crear_llm("google/gemini-2.0-flash-exp", 0.7)

# Opcion 3: Especializado tecnico
llm_investigador = crear_llm("qwen/qwen-2.5-7b-instruct", 0.3)
llm_redactor = crear_llm("microsoft/phi-3.5-mini-128k-instruct", 0.7)

# Opcion 4: Codigo y matematicas
llm_investigador = crear_llm("deepseek/deepseek-v2.5", 0.2)
llm_redactor = crear_llm("qwen/qwen-2.5-7b-instruct", 0.5)

# Opcion 5: Creatividad
llm_investigador = crear_llm("mistralai/mistral-7b-instruct", 0.4)
llm_redactor = crear_llm("nousresearch/hermes-3-llama-3.1-8b", 0.8)

# Opcion 6: Rapido y ligero
llm_investigador = crear_llm("google/gemini-2.0-flash-lite-preview-02-05", 0.5, 2000)
llm_redactor = crear_llm("qwen/qwen-2.5-3b-instruct", 0.6, 2000)
```

## Recomendaciones por Tipo de Tarea

| Tarea | Modelo Recomendado | Temperature |
|-------|-------------------|-------------|
| Investigación científica | Gemini 2.0 Flash | 0.3 |
| Análisis de código | DeepSeek V2.5 | 0.2 |
| Redacción técnica | Phi-3.5 Mini | 0.7 |
| Escritura creativa | Hermes 3 | 0.8 |
| Conversación | Mistral 7B | 0.6 |
| Procesamiento rápido | Gemini Flash Lite | 0.5 |
| Matemáticas | Qwen 2.5 7B | 0.3 |
| Documentación | Phi-3 Mini | 0.6 |


```python

# Opción 1: Ambos con Gemini (muy capaz)
llm_gemini = ChatOpenAI(
    model="google/gemini-2.0-flash-exp",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.5,
    max_tokens=4000,
    base_url="https://openrouter.ai/api/v1"
)

# Opción 2: Investigador con Qwen 2.5 (bueno para análisis técnico)
llm_qwen = ChatOpenAI(
    model="qwen/qwen-2.5-7b-instruct",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.3,
    max_tokens=4000,
    base_url="https://openrouter.ai/api/v1"
)

# Opción 3: Redactor con Phi-3.5 (bueno para texto claro)
llm_phi = ChatOpenAI(
    model="microsoft/phi-3.5-mini-128k-instruct",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.7,
    max_tokens=3000,
    base_url="https://openrouter.ai/api/v1"
)

```


Todos estos modelos son **completamente gratuitos** en OpenRouter y no requieren créditos ni pagos.

### 3.2 Task — description, expected_output

Una `Task` define **qué hacer** (description) y **qué formato de output se espera** (expected_output). El campo `expected_output` actúa como el criterio de evaluación interno del agente — si el output no lo cumple, el agente intenta de nuevo.

Las tasks pueden tener **context**: lista de otras tasks cuyo output se pasa como entrada. Esto crea dependencias explícitas entre tareas.

In [5]:
# Paso 3: Definir tasks con contexto entre ellas

tarea_investigacion = Task(
    description=(
        "Realiza una revisión del estado del arte sobre el uso de IA (especialmente sistemas multi-agente) "
        "en la investigación de nanomateriales. "
        "Cubre: (1) técnicas de ML para predicción de propiedades, "
        "(2) optimización de síntesis con RL, "
        "(3) análisis automatizado de imágenes TEM/SEM, "
        "(4) agentes para revisión de literatura."
    ),
    expected_output=(
        "Un informe estructurado de 400-600 palabras con: "
        "(1) resumen ejecutivo de 3-4 oraciones, "
        "(2) estado del arte en 4 subsecciones con las áreas indicadas, "
        "(3) identificación de 2-3 gaps de investigación, "
        "(4) al menos 5 referencias específicas (autor, año, tema)."
    ),
    agent=investigador,
)

tarea_redaccion = Task(
    description=(
        "Basándote en el informe de revisión del investigador, "
        "crea un artículo de divulgación técnica de 300-400 palabras "
        "que explique el estado del arte a una audiencia de ingenieros de materiales "
        "sin experiencia en IA. "
        "Incluye: título atractivo, introducción, 3 secciones temáticas, implicaciones prácticas."
    ),
    expected_output=(
        "Un artículo de divulgación con título, 4 secciones claramente marcadas con headers, "
        "sin jerga de IA innecesaria, con al menos un ejemplo concreto en cada sección, "
        "y una sección final 'Implicaciones Prácticas' de 3+ bullet points."
    ),
    agent=redactor,
    context=[tarea_investigacion],  # el output de investigación es input aquí
)

print("Tasks definidas:")
print(f"  - tarea_investigacion (agente: {tarea_investigacion.agent.role})")
print(f"  - tarea_redaccion (agente: {tarea_redaccion.agent.role}, contexto: tarea_investigacion)")

Tasks definidas:
  - tarea_investigacion (agente: Investigador Senior en IA y Nanotecnologia)
  - tarea_redaccion (agente: Redactor Tecnico Senior, contexto: tarea_investigacion)


### 3.3 Crew y Process: Sequential vs Hierarchical

In [6]:
# ────────────────────────────────────────────────────────────
# Output esperado: crew secuencial investigador → redactor
# ────────────────────────────────────────────────────────────

# Paso 4: Crear la Crew con Process.sequential
# Sequential = las tasks se ejecutan en el orden en que se definieron
crew_secuencial = Crew(
    agents=[investigador, redactor],
    tasks=[tarea_investigacion, tarea_redaccion],
    process=Process.sequential,
    verbose=True,
)

print("Crew secuencial configurada:")
print(f"  Agentes: {len(crew_secuencial.agents)}")
print(f"  Tasks: {len(crew_secuencial.tasks)}")
print(f"  Proceso: Sequential")

Crew secuencial configurada:
  Agentes: 2
  Tasks: 2
  Proceso: Sequential


In [7]:
# Paso 5: Ejecutar la Crew
# NOTA: Esta celda hace llamadas reales al LLM — puede tardar 30-90 segundos
print("Iniciando misión de investigación...")
print("=" * 60)

resultado_secuencial = crew_secuencial.kickoff()

print("\n" + "=" * 60)
print("RESULTADO FINAL DE LA CREW:")
print("=" * 60)
print(resultado_secuencial.raw)

Iniciando misión de investigación...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ffb34ea7-2482-41ca-b789-7cb56ac45d0c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Realiza una revisión del estado del arte sobre el uso de IA (especialmente sistemas multi-agente) en la  │
│  investigación de nanomateriales. Cubre: (1) técnicas de ML para predicción de propiedades, (2) optimización    │
│  de síntesis con RL, (3) análisis automatizado de imágenes TEM/SEM, (4) agentes para revisión de literatura.    │
│  ID: d32af182-84a1-4595-9521-cad5b53c4cce                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Query Error ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Query Failed                                                                                            │
│  Source: Unified Memory                                                                                         │
│  Error: Memory requires an LLM for analysis but initialization failed: Error importing native provider:         │
│  OPENAI_API_KEY is required                                                                                     │
│                                                                                                                 │
│  To fix this, do one of the following:                                                                          │
│    - Set OPENAI_API_KEY for the default model (gpt-4o-mini)                                                     │
│    - Pass a different model: Memory(llm="anthropic/claude-3-haiku-20240307")                                    │
│    - Pass any LLM instance: Memory(llm=LLM(model="your-model"))                                                 │
│    - To skip LLM analysis, pass all fields explicitly to remember()                                             │
│      and use depth="shallow" for recall.                                                                        │
│                                                                                                                 │
│  Docs: https://docs.crewai.com/concepts/memory                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


[2026-05-13 19:37:00][ERROR]: Failed to save to memory: Memory requires an LLM for analysis but initialization failed: Error importing native provider: OPENAI_API_KEY is required

To fix this, do one of the following:
  - Set OPENAI_API_KEY for the default model (gpt-4o-mini)
  - Pass a different model: Memory(llm="anthropic/claude-3-haiku-20240307")
  - Pass any LLM instance: Memory(llm=LLM(model="your-model"))
  - To skip LLM analysis, pass all fields explicitly to remember()
    and use depth="shallow" for recall.

Docs: https://docs.crewai.com/concepts/memory


╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Basándote en el informe de revisión del investigador, crea un artículo de divulgación técnica de         │
│  300-400 palabras que explique el estado del arte a una audiencia de ingenieros de materiales sin experiencia   │
│  en IA. Incluye: título atractivo, introducción, 3 secciones temáticas, implicaciones prácticas.                │
│  Agent: Redactor Tecnico Senior                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ffb34ea7-2482-41ca-b789-7cb56ac45d0c                                                                       │
│  Final Output: **Título:** Estado del Arte en Ingeniería de Materiales: Técnicas Avanzadas para la              │
│  Optimización y Análisis                                                                                        │
│                                                                                                                 │
│  **Introducción**                                                                                               │
│                                                                                                                 │
│  La ingeniería de materiales es un campo en constante evolución, donde la aplicación de técnicas de             │
│  inteligencia artificial (IA) está revolucionando la forma en que se diseñan, sintetizan y analizan             │
│  materiales. En este artículo, exploraremos el estado del arte en tres áreas clave: técnicas de Machine         │
│  Learning (ML) para la predicción de propiedades, optimización de síntesis con Reinforcement Learning (RL),     │
│  análisis automatizado de imágenes TEM/SEM y agentes para la revisión de literatura.                            │
│                                                                                                                 │
│  **Técnicas de ML para la Predicción de Propiedades**                                                           │
│                                                                                                                 │
│  Las técnicas de Machine Learning han demostrado ser efectivas en la predicción de propiedades de materiales.   │
│  Por ejemplo, los modelos de redes neuronales profundas pueden aprender patrones en grandes conjuntos de datos  │
│  y predecir con precisión propiedades como la dureza, la resistencia a la corrosion o la conductividad          │
│  térmica. Un ejemplo concreto es el uso de redes neuronales para predecir la dureza de aceros aleados.          │
│  Mediante la entrenamiento de un modelo con datos de propiedades y composición de aceros, se puede mejorar      │
│  significativamente la precisión de la predicción en comparación con métodos tradicionales.                     │
│                                                                                                                 │
│  **Optimización de Síntesis con RL**                                                                            │
│                                                                                                                 │
│  La optimización de síntesis con Reinforcement Learning ha demostrado ser efectiva en la búsqueda de            │
│  condiciones óptimas para la síntesis de materiales. Por ejemplo, un agente de RL puede aprender a regular la   │
│  temperatura, la presión y la composición de un reactor para producir un material con propiedades específicas.  │
│  Un estudio reciente demostró que un agente de RL logró optimizar la síntesis de nanomateriales con una mejora  │
│  del 30% en la eficiencia de producción.                                                                        │
│                                                                                                                 │
│  **Análisis Automatizado de Imágenes TEM/SEM**                                                                  │
│                                                       


RESULTADO FINAL DE LA CREW:
**Título:** Estado del Arte en Ingeniería de Materiales: Técnicas Avanzadas para la Optimización y Análisis

**Introducción**

La ingeniería de materiales es un campo en constante evolución, donde la aplicación de técnicas de inteligencia artificial (IA) está revolucionando la forma en que se diseñan, sintetizan y analizan materiales. En este artículo, exploraremos el estado del arte en tres áreas clave: técnicas de Machine Learning (ML) para la predicción de propiedades, optimización de síntesis con Reinforcement Learning (RL), análisis automatizado de imágenes TEM/SEM y agentes para la revisión de literatura.

**Técnicas de ML para la Predicción de Propiedades**

Las técnicas de Machine Learning han demostrado ser efectivas en la predicción de propiedades de materiales. Por ejemplo, los modelos de redes neuronales profundas pueden aprender patrones en grandes conjuntos de datos y predecir con precisión propiedades como la dureza, la resistencia a la corro

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Output esperado:** Primero verás el `investigador` produciendo el informe de revisión (con `Thought:` y contenido). Luego el `redactor` toma ese informe como contexto y produce el artículo de divulgación. Nótese que el redactor tiene acceso al output del investigador automáticamente via el campo `context=[tarea_investigacion]`.

**Interpretación:** El parámetro `memory=True` en el agente investigador hace que recuerde el contexto de su tarea cuando el redactor le pide aclaraciones. La `backstory` impacta directamente en el tono y el nivel de detalle del output — el investigador cita referencias específicas, el redactor incluye la sección de implicaciones prácticas, tal como sus backstories les instruyen.

---
## 4. Tools en CrewAI <a id='4-tools'></a>

### 4.1 Tools Nativas de CrewAI

CrewAI incluye tools pre-construidas como `SerperDevTool` (búsqueda web), `FileReadTool`, `CSVSearchTool`, entre otras. Requieren sus respectivas API keys.

### 4.2 Bridging: Tools de LangChain en CrewAI

Una de las ventajas clave de CrewAI es que acepta tools definidas con el decorador `@tool` de LangChain directamente.

In [8]:
# ────────────────────────────────────────────────────────────
# Output esperado: agente CrewAI usando tools nativas de CrewAI
# ────────────────────────────────────────────────────────────
import math
from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

# Nota: CrewAI 0.80.0 requiere crewai.tools.BaseTool — no acepta
# langchain_core StructuredTool directamente en Agent.tools.

# ============================================================
# Paso 1: Schemas de input (requerido por BaseTool)
# ============================================================

class BuscarNanoInput(BaseModel):
    material: str = Field(description="Nombre del material (ej: 'oro', 'grafeno', 'plata')")

class EnergiaInput(BaseModel):
    radio_nm: float = Field(description="Radio de la nanopartícula en nanómetros")
    material: str = Field(description="Nombre del material (ej: 'oro', 'dioxido de titanio')")

# ============================================================
# Paso 2: Definir tools heredando de crewai.tools.BaseTool
# ============================================================

class BuscarDatosNanoparticulas(BaseTool):
    name: str = "buscar_datos_nanoparticulas"
    description: str = (
        "Busca propiedades físico-químicas de una nanopartícula o material dado. "
        "Útil cuando necesitas datos sobre tamaño típico, propiedades ópticas o magnéticas. "
        "Input: nombre del material (ej: 'oro', 'dióxido de titanio', 'grafeno')"
    )
    args_schema: Type[BaseModel] = BuscarNanoInput

    def _run(self, material: str) -> str:
        db = {
            "oro": {
                "tamaño_tipico": "1-100 nm",
                "propiedades": "SPR en ~520nm, biocompatible, alta estabilidad química",
                "aplicaciones": "biomedicina, catálisis, sensores",
            },
            "dioxido de titanio": {
                "tamaño_tipico": "5-50 nm",
                "propiedades": "fotocatálisis UV, blanco intenso, alta refracción",
                "aplicaciones": "protector solar, fotovoltaica, pinturas",
            },
            "grafeno": {
                "tamaño_tipico": "monocapa ~0.3nm",
                "propiedades": "conductividad eléctrica 1M S/m, resistencia mecánica 130 GPa",
                "aplicaciones": "electrónica, compuestos, baterías",
            },
            "plata": {
                "tamaño_tipico": "5-100 nm",
                "propiedades": "antibacteriano, SERS, SPR en ~400nm",
                "aplicaciones": "antimicrobiano, diagnóstico, tintas conductoras",
            },
        }
        m = material.lower()
        for key, data in db.items():
            if key in m or m in key:
                return (
                    f"Material: {key}\n"
                    f"Tamaño: {data['tamaño_tipico']}\n"
                    f"Propiedades: {data['propiedades']}\n"
                    f"Aplicaciones: {data['aplicaciones']}"
                )
        return f"Material '{material}' no encontrado. Disponibles: {', '.join(db.keys())}"


class CalcularEnergiaSurface(BaseTool):
    name: str = "calcular_energia_surface"
    description: str = (
        "Calcula la energía de superficie aproximada de una nanopartícula esférica. "
        "Aplica la aproximación: E_surf = 4πr² * γ donde γ es la energía de superficie específica. "
        "Inputs: radio_nm (radio en nanómetros), material (nombre del material)"
    )
    args_schema: Type[BaseModel] = EnergiaInput

    def _run(self, radio_nm: float, material: str) -> str:
        gamma_db = {"oro": 1.50, "plata": 1.20, "dioxido de titanio": 0.90, "grafeno": 0.05}
        gamma = gamma_db.get(material.lower(), 1.0)
        radio_m = radio_nm * 1e-9
        area = 4 * math.pi * radio_m ** 2
        energia = area * gamma
        return (
            f"Radio: {radio_nm} nm = {radio_m:.2e} m\n"
            f"Área superficial: {area:.4e} m²\n"
            f"γ ({material}): {gamma} J/m²\n"
            f"Energía de superficie: {energia:.4e} J"
        )


# ============================================================
# Paso 3: Instanciar las tools e inyectarlas en el agente
# ============================================================
buscar_tool = BuscarDatosNanoparticulas()
energia_tool = CalcularEnergiaSurface()

analista_nano = Agent(
    role="Analista de Nanomateriales",
    goal="Analizar propiedades físico-químicas de nanomateriales y realizar cálculos de energía de superficie.",
    backstory=(
        "Especialista en caracterización de nanomateriales con experiencia en AFM, TEM y DLS. "
        "Rigoroso en los cálculos y siempre presenta resultados con unidades correctas y contexto físico."
    ),
    tools=[buscar_tool, energia_tool],
    llm=llm,
    verbose=True,
    max_iter=4,
)

tarea_analisis = Task(
    description=(
        "Análisis de nanopartículas de oro:\n"
        "1. Busca las propiedades del oro nanométrico\n"
        "2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm\n"
        "3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa"
    ),
    expected_output=(
        "Un análisis que incluya: propiedades del nanoro obtenidas de la base de datos, "
        "tabla comparativa de energías de superficie para 3 tamaños, "
        "interpretación física del efecto de escala en al menos 2 párrafos."
    ),
    agent=analista_nano,
)

crew_analista = Crew(
    agents=[analista_nano],
    tasks=[tarea_analisis],
    process=Process.sequential,
    verbose=True,
)

print("Tools CrewAI definidas:")
print(f"  - {buscar_tool.name}")
print(f"  - {energia_tool.name}")
print("Crew de analista configurada.")


Tools CrewAI definidas:
  - buscar_datos_nanoparticulas
  - calcular_energia_surface
Crew de analista configurada.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [9]:
# Ejecutar el análisis
resultado_analisis = crew_analista.kickoff()
print("\n" + "=" * 60)
print("ANÁLISIS DE NANOPARTÍCULAS:")
print(resultado_analisis.raw)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.11.0                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: fede9bb1-346d-4b27-b269-88f206429b16                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Análisis de nanopartículas de oro:                                                                       │
│  1. Busca las propiedades del oro nanométrico                                                                   │
│  2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm                                            │
│  3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa                           │
│  ID: d900f35d-4288-4bc6-b9b1-ccfe53ed5977                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Nanomateriales                                                                              │
│                                                                                                                 │
│  Task: Análisis de nanopartículas de oro:                                                                       │
│  1. Busca las propiedades del oro nanométrico                                                                   │
│  2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm                                            │
│  3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 403 - Your project has been denied access. Please contact support.


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Análisis de nanopartículas de oro:                                                                       │
│  1. Busca las propiedades del oro nanométrico                                                                   │
│  2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm                                            │
│  3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa                           │
│  Agent: Analista de Nanomateriales                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: fede9bb1-346d-4b27-b269-88f206429b16                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

---
## 5. Process Hierarchical: Manager + Delegates <a id='5-hierarchical'></a>

En `Process.sequential` las tareas se ejecutan en orden fijo. En `Process.hierarchical`, un **agente manager** decide dinámicamente qué agente realiza cada tarea y en qué orden, basándose en las instrucciones y el estado del proyecto.

```
SEQUENTIAL:        Manager  (solo orquesta)
  tarea1 → a1     /     \
  tarea2 → a2    a1     a2
  tarea3 → a3    |      |
                t1     t2/t3 (decide el manager)

HIERARCHICAL:
  Manager decide: "a1 hace t1, luego a2 decide si hace t2 o t3 según el resultado de t1"
```

In [ ]:

# ────────────────────────────────────────────────────────────
# Output esperado: crew jerárquica con manager dinámico
# ────────────────────────────────────────────────────────────

# Paso 1: Definir el agente manager
# Con Process.hierarchical, CrewAI crea un manager interno
# O podemos definir uno explícito con manager_agent

director_proyecto = Agent(
    role="Director de Proyecto de Investigación",
    goal=(
        "Coordinar al equipo de investigación para producir un informe técnico de alta calidad. "
        "Delegar tareas a los especialistas adecuados y asegurarse de que cada output cumple los estándares."
    ),
    backstory=(
        "Eres un director con 20 años de experiencia liderando proyectos de I+D en nanotecnología. "
        "Sabes identificar qué especialista es el más adecuado para cada tarea. "
        "Eres exigente con la calidad pero eficiente — no pides trabajo innecesario. "
        "Cuando el output de un especialista no cumple los criterios, lo devuelves con feedback específico."
    ),
    llm=llm,
    allow_delegation=True,  # puede delegar a otros agentes
    verbose=True,
)

# Paso 2: Crew jerárquica
# Task única de alto nivel — el manager la descompone y delega
tarea_proyecto = Task(
    description=(
        "Producir un informe técnico completo sobre 'Aplicaciones de IA en síntesis de nanopartículas'. "
        "El informe debe incluir: revisión del estado del arte, análisis de propiedades de al menos 2 materiales, "
        "y una sección de divulgación accesible para no-especialistas."
    ),
    expected_output=(
        "Informe estructurado con: portada, índice implícito, sección técnica (800+ palabras), "
        "análisis de materiales con datos numéricos, y sección de divulgación (300+ palabras). "
        "Total: 1100+ palabras con nivel de calidad publicable."
    ),
    agent=director_proyecto,
)

# NOTA CrewAI 0.80.0: el manager_agent NO debe incluirse en agents[]
# El manager orquesta pero no es un worker — va solo en manager_agent=
crew_jerarquica = Crew(
    agents=[investigador, analista_nano, redactor],  # solo workers
    tasks=[tarea_proyecto],
    process=Process.hierarchical,
    manager_agent=director_proyecto,
    verbose=True,
)

print("Crew jerárquica configurada:")
print(f"  Manager: {director_proyecto.role}")
print(f"  Equipo: {[a.role for a in [investigador, analista_nano, redactor]]}")


In [ ]:
# NOTA: Process.hierarchical hace más llamadas al LLM que Sequential
# Es más lento pero produce outputs de mayor calidad en tareas complejas
# Tiempo estimado: 2-4 minutos con Gemini Flash

print("Iniciando proyecto con crew jerárquica...")
print("El director asignará subtareas dinámicamente a los especialistas.")
print("=" * 60)

resultado_jerarquico = crew_jerarquica.kickoff()

print("\n" + "=" * 60)
print("INFORME FINAL DEL DIRECTOR:")
print("=" * 60)
print(resultado_jerarquico.raw[:1000], "...[continúa]")

**Interpretación:** En `Process.hierarchical`, el director decide en tiempo de ejecución si delega al investigador primero, luego al analista, luego al redactor — o si los llama en un orden diferente según los resultados intermedios. Esto no es posible en `Process.sequential` donde el orden es fijo. La diferencia es visible en los logs: con hierarchical verás al manager evaluando outputs y tomando decisiones de delegación explícitas.

---
## 6. Skill: Output Scorer — Evaluación Automática <a id='6-output-scorer'></a>

La skill `output_scorer` cierra el ciclo de calidad: permite evaluar automáticamente el output de cualquier agente o crew usando criterios definidos programáticamente.

In [ ]:
# ────────────────────────────────────────────────────────────
# Output esperado: evaluar el informe generado por la crew
# ────────────────────────────────────────────────────────────

# Paso 1: Cargar la skill (con reload para limpiar caché del kernel)
import importlib
import sys

mods_to_reload = [m for k, m in sys.modules.items() if "output_scorer" in k]
for mod in mods_to_reload:
    importlib.reload(mod)

from external_skills.evaluation.output_scorer import (
    score_with_llm, score_heuristic, DEFAULT_CRITERIA, ScoreResult, SKILL_METADATA as OS_META
)

print(f"Skill cargada: {OS_META['name']} v{OS_META['version']}")
print(f"Criterios por defecto: {list(DEFAULT_CRITERIA.keys())}")


In [ ]:
# Paso 2: Evaluación heurística del informe (sin costo de LLM)
score_heuristic_result = score_heuristic(
    output=resultado_secuencial.raw,
    task_description="Artículo de divulgación técnica sobre IA en nanotecnología"
)

print("Evaluación Heurística:")
print(f"  Score: {score_heuristic_result.score:.2f}/{score_heuristic_result.max_score}")
print(f"  Dimensiones:")
for dim, val in score_heuristic_result.breakdown.items():
    print(f"    {dim}: {val:.2f}")
print(f"  Feedback: {score_heuristic_result.feedback[:200]}")

In [ ]:
# Paso 3: Evaluación con LLM (más detallada)
# Criterios personalizados para este tipo de output
criterios_articulo = {
    "rigor_tecnico": 0.30,      # precisión de contenido técnico
    "claridad": 0.25,           # accesibilidad para no-especialistas
    "estructura": 0.20,         # organización y formato
    "completitud": 0.25,        # cubre todas las áreas requeridas
}

print("Iniciando evaluación con LLM (puede tardar 5-10 segundos)...")
score_llm_result = score_with_llm(
    output=resultado_secuencial.raw,
    task_description="Artículo de divulgación sobre IA en nanotecnología para ingenieros de materiales",
    criteria=criterios_articulo,
    llm=llm
)

print("\nEvaluación con LLM:")
print(f"  Score total: {score_llm_result.score:.2f}/{score_llm_result.max_score}")
print(f"  Dimensiones:")
for dim, val in score_llm_result.breakdown.items():
    bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
    print(f"    {dim:<20} {bar} {val:.2f}")
print(f"\n  Feedback completo:")
print(f"  {score_llm_result.feedback}")

**Interpretación:** Tener scoring automático transforma la evaluación de agentes de una actividad subjetiva en una métrica medible. Con `score_heuristic` podemos verificar rápidamente si el output cumple requisitos básicos (longitud, estructura, keywords). Con `score_with_llm` obtenemos una evaluación semántica más rica. En un pipeline de producción, esta skill se usaría como nodo de quality gate en el grafo LangGraph — si el score es menor a 0.7, el output se rechaza y se regenera.

---
## 7. Proyecto: Equipo de Investigación en Nanotecnología <a id='7-proyecto'></a>

Integración completa: crew de 4 agentes especializados + tools de dominio + evaluación automática con output_scorer.

In [ ]:
# ============================================================
# PROYECTO: Equipo Completo de Investigación en Nanomateriales
# ============================================================

# Paso 1: Definir el equipo completo

revisor_calidad = Agent(
    role="Revisor de Calidad Científica",
    goal=(
        "Verificar que el informe final cumple estándares de rigor científico, "
        "consistencia interna, y precisión técnica. "
        "Identificar afirmaciones sin respaldo y recomendar correcciones."
    ),
    backstory=(
        "Ex-editor de ACS Nano con 10 años de experiencia en peer review. "
        "Tienes ojo clínico para detectar over-claims, datos incorrectos y falta de rigor. "
        "Tu review es siempre constructivo: señalas el problema Y propones la corrección específica. "
        "No apruebas un documento si contiene errores factuales o afirmaciones sin evidencia."
    ),
    llm=llm,
    verbose=True,
    max_iter=2,
)

# Paso 2: Pipeline de tareas con dependencias en cadena
# investigacion → analisis → redaccion → revision

t1_investigacion = Task(
    description=(
        "Investiga el estado actual de la síntesis de nanopartículas de oro usando métodos asistidos por IA. "
        "Enfócate en: (1) optimización por reinforcement learning, "
        "(2) predicción de propiedades por ML, (3) control de calidad automatizado. "
        "Incluye 3+ referencias específicas."
    ),
    expected_output="Informe técnico de 300-400 palabras con 3 subsecciones temáticas y 3+ referencias.",
    agent=investigador,
)

t2_analisis = Task(
    description=(
        "Basándote en el informe de investigación, "
        "realiza un análisis cuantitativo de las propiedades de las nanopartículas de oro. "
        "Calcula la energía de superficie para radios de 2nm, 5nm y 10nm. "
        "Busca las propiedades en la base de datos y presenta los resultados en formato de tabla."
    ),
    expected_output=(
        "Análisis cuantitativo con: propiedades del nanoro de la base de datos, "
        "tabla de energías para 3 tamaños, interpretación física en 1 párrafo."
    ),
    agent=analista_nano,
    context=[t1_investigacion],
)

t3_redaccion = Task(
    description=(
        "Integra el informe de investigación y el análisis cuantitativo en un "
        "documento técnico coherente de 600-800 palabras. "
        "Audiencia: ingenieros de producción en una empresa de nanomateriales. "
        "Incluye: resumen ejecutivo (5 líneas), cuerpo técnico, tabla de datos, conclusiones accionables."
    ),
    expected_output=(
        "Documento integrado con resumen ejecutivo, 3 secciones técnicas, "
        "al menos 1 tabla de datos, y 3+ conclusiones accionables en bullet points."
    ),
    agent=redactor,
    context=[t1_investigacion, t2_analisis],
)

t4_revision = Task(
    description=(
        "Revisa el documento integrado en busca de: inconsistencias entre secciones, "
        "afirmaciones sin respaldo, errores en los cálculos de energía, y problemas de claridad. "
        "Proporciona el documento corregido o una lista de correcciones específicas."
    ),
    expected_output=(
        "Lista numerada de encontrados (o 'APROBADO' si no hay problemas) + "
        "la versión final corregida del documento completo."
    ),
    agent=revisor_calidad,
    context=[t3_redaccion],
)

# Paso 3: Crew del proyecto completo
crew_investigacion = Crew(
    agents=[investigador, analista_nano, redactor, revisor_calidad],
    tasks=[t1_investigacion, t2_analisis, t3_redaccion, t4_revision],
    process=Process.sequential,
    verbose=True,
)

# Mapa explícito de LLMs por agente
_llm_labels = {
    investigador.role:    "openrouter/google/gemini-2.0-flash-001",
    analista_nano.role:   llm.model,
    redactor.role:        "openrouter/meta-llama/llama-3.1-8b-instruct",
    revisor_calidad.role: llm.model,
}

print("Equipo completo de investigación configurado:")
print(f"  {'Agente':<40} {'LLM'}")
print(f"  {'-'*40} {'-'*40}")
for a in crew_investigacion.agents:
    print(f"  {a.role:<40} {_llm_labels[a.role]}")
print(f"\nPipeline: t1 Investigación → t2 Análisis → t3 Redacción → t4 Revisión")


In [ ]:
# Paso 4: Ejecutar el proyecto completo
# NOTA: Esto hace 4 llamadas encadenadas al LLM — puede tardar 3-5 minutos
print("Ejecutando proyecto de investigación completo...")
print("Pipeline: Investigación → Análisis → Redacción → Revisión")
print("=" * 60)

resultado_proyecto = crew_investigacion.kickoff()

print("\n" + "=" * 60)
print("DOCUMENTO FINAL REVISADO:")
print("=" * 60)
print(resultado_proyecto.raw)

In [ ]:
# Paso 5: Evaluar el output final con output_scorer
evaluacion_final = score_with_llm(
    output=resultado_proyecto.raw,
    task_description=(
        "Documento técnico sobre síntesis de nanopartículas asistida por IA, "
        "para ingenieros de producción, con datos cuantitativos y conclusiones accionables"
    ),
    criteria={
        "rigor_tecnico": 0.25,
        "datos_cuantitativos": 0.25,
        "claridad_audiencia": 0.20,
        "accionabilidad": 0.20,
        "coherencia_interna": 0.10,
    },
    llm=llm
)

print("\nEvaluación del Proyecto:")
print(f"Score: {evaluacion_final.score:.2f}/{evaluacion_final.max_score} ",
      end="")
pct = evaluacion_final.score / evaluacion_final.max_score
grade = "EXCELENTE" if pct > 0.85 else "BUENO" if pct > 0.70 else "ACEPTABLE" if pct > 0.55 else "INSUFICIENTE"
print(f"({grade})")
print()
print("Breakdown:")
for dim, val in evaluacion_final.breakdown.items():
    bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
    print(f"  {dim:<25} {bar} {val:.2f}")
print(f"\nFeedback:\n{evaluacion_final.feedback}")

---
## 8. Resumen y Criterios de Evaluación <a id='8-resumen'></a>

### Conceptos Clave de esta Notebook

| Concepto | Definición | Implementado en |
|----------|------------|------------------|
| **Agent** | Entidad con role, goal y backstory que define su expertise y comportamiento | Sección 3.1 |
| **Task** | Unidad de trabajo con descripción, expected_output y contexto de tareas previas | Sección 3.2 |
| **Process.sequential** | Tasks ejecutadas en orden prefijado | Sección 3.3 |
| **Process.hierarchical** | Manager asigna tasks dinámicamente según resultados | Sección 5 |
| **Tool bridging** | Tools `@tool` de LangChain funcionan directamente en CrewAI | Sección 4.2 |
| **context=[]** | Pasa el output de una task como input de otra (dependencia explícita) | Secciones 3.2, 7 |
| **Output Scorer** | Skill para evaluación automática de calidad de outputs multi-agente | Sección 6 |

### Criterios de Evaluación

- [ ] Definir 2+ agentes con role, goal y backstory diferenciados y verificar el impacto en los outputs
- [ ] Construir un pipeline de 3+ tasks con dependencias `context=[...]`
- [ ] Ejecutar el mismo pipeline con `Process.sequential` y describir la diferencia con hierarchical
- [ ] Usar `output_scorer` para evaluación automática y obtener un score > 0.7 ajustando los prompts
- [ ] Demostrar tool bridging: una tool LangChain usada por un agente CrewAI

### Ejercicio de Extensión

Construye una crew de 4 agentes para producir un plan de negocio sobre comercialización de un nanomaterial:
1. **Científico** — viable técnicamente (con tools de cálculo de propiedades)
2. **Economista** — análisis de mercado y costos
3. **Regulatorio** — aspectos legales y de seguridad nanotecnológica
4. **Redactor Ejecutivo** — integra todo en un executive summary de 1 página

Usa `output_scorer` con criterios personalizados para el dominio de negocio y ajusta las backstories hasta obtener un score > 0.85.

### Próxima Notebook

En **U5_04 — Google ADK y Protocolo A2A** exploraremos el framework oficial de Google para agentes, el protocolo Agent-to-Agent (A2A) para comunicación entre agentes de diferentes frameworks, y la integración con Gemini como LLM principal.

---
*Notebook generada siguiendo el PROTOCOLO_UNIDAD_5_MULTIAGENTE.md v1.1.0*  
*Antigravity-Nano-Research — Unit 05 Multi-Agent Systems — Marzo 2026*

---
## Ejercicio de Extensión: Plan de Negocio — Clasificador SEM como Servicio

Construimos una **crew de 4 agentes** para producir un plan de negocio sobre la comercialización
del clasificador de micrografías SEM (ResNet-18) como servicio SaaS para laboratorios de nanotecnología.

| # | Agente | Rol | Función |
|---|--------|-----|---------|
| 1 | **Científico** | Viabilidad técnica | Valida que el modelo (99.07% accuracy) es competitivo y define limitaciones |
| 2 | **Economista** | Análisis de mercado | Estima TAM, pricing y costos operativos |
| 3 | **Regulatorio** | Aspectos legales | Analiza normativa de nanomateriales y requisitos de certificación |
| 4 | **Redactor Ejecutivo** | Integración | Produce un executive summary de 1 página |

Al final, evaluamos con `output_scorer` usando criterios de dominio de negocio.

In [2]:
# ============================================================
# EJERCICIO: CREW DE 4 AGENTES — PLAN DE NEGOCIO SEM
# ============================================================
import os, logging
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

logging.getLogger("litellm").setLevel(logging.ERROR)
load_dotenv(override=True)

openrouter_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_key:
    raise ValueError("OPENROUTER_API_KEY no encontrada en .env")

# --- Tool: Métricas del Modelo SEM ---
class SEMModelMetricsTool(BaseTool):
    name: str = "sem_model_metrics"
    description: str = (
        "Obtiene las metricas de rendimiento del clasificador SEM ResNet-18. "
        "Usa esta tool para validar la viabilidad tecnica del producto."
    )
    def _run(self, query: str = "") -> str:
        return (
            "Metricas del Clasificador SEM ResNet-18:\n"
            "- Accuracy: 99.07% (106/107 correctas)\n"
            "- F1-Score: 0.9903\n"
            "- AUC-ROC: 0.9943\n"
            "- Clases: Nanoparticles (0D) vs Nanowires (1D)\n"
            "- Dataset: NFFA-EUROPE SEM (535 imagenes)\n"
            "- Arquitectura: ResNet-18 fine-tuned desde ImageNet\n"
            "- Interpretabilidad: Grad-CAM en layer4[-1]\n"
            "- Tiempo de inferencia: ~50ms por imagen (GPU)\n"
            "- Error: 1/107 (una nanoparticula clasificada como nanohilo)"
        )

sem_metrics_tool = SEMModelMetricsTool()
print("Tool SEM Metrics definida.")
print(sem_metrics_tool._run())

Tool SEM Metrics definida.
Metricas del Clasificador SEM ResNet-18:
- Accuracy: 99.07% (106/107 correctas)
- F1-Score: 0.9903
- AUC-ROC: 0.9943
- Clases: Nanoparticles (0D) vs Nanowires (1D)
- Dataset: NFFA-EUROPE SEM (535 imagenes)
- Arquitectura: ResNet-18 fine-tuned desde ImageNet
- Interpretabilidad: Grad-CAM en layer4[-1]
- Tiempo de inferencia: ~50ms por imagen (GPU)
- Error: 1/107 (una nanoparticula clasificada como nanohilo)


In [3]:
# ============================================================
# DEFINICIÓN DE LOS 4 AGENTES
# ============================================================

MODELS_FALLBACK = [
    "google/gemini-2.5-flash",
    "openai/gpt-4o-mini",
    "anthropic/claude-3-haiku",
]

def crear_llm(temperature=0.3, max_tokens=3000):
    for model_id in MODELS_FALLBACK:
        try:
            _c = LLM(
                model=f"openrouter/{model_id}",
                api_key=openrouter_key,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            _c.call([{"role": "user", "content": "OK"}])
            return _c, model_id
        except Exception:
            continue
    raise RuntimeError("No hay modelos disponibles")

llm_obj, model_name = crear_llm(temperature=0.2)
print(f"LLM: {model_name}")

# --- AGENTE 1: CIENTÍFICO ---
cientifico = Agent(
    role="Cientifico Senior en Vision Artificial y Nanomateriales",
    goal=(
        "Evaluar la viabilidad tecnica del clasificador SEM ResNet-18 como producto "
        "comercial, identificando fortalezas, limitaciones y mejoras necesarias."
    ),
    backstory=(
        "Eres un investigador con 12 anos de experiencia en computer vision aplicada a "
        "microscopia electronica. Has desarrollado modelos de clasificacion para TEM, SEM y AFM "
        "en laboratorios de materiales avanzados. Conoces el estado del arte en transfer learning "
        "para imagenes cientificas y puedes evaluar si un modelo es competitivo frente a alternativas "
        "comerciales como Thermo Fisher Avizo o MIPAR. Siempre cuantificas tus argumentos con metricas."
    ),
    llm=llm_obj,
    tools=[sem_metrics_tool],
    verbose=True,
    max_iter=3,
    memory=False,
    allow_delegation=False,
)

# --- AGENTE 2: ECONOMISTA ---
economista = Agent(
    role="Analista de Mercado en Tecnologia Deep Tech",
    goal=(
        "Realizar un analisis de mercado para un servicio SaaS de clasificacion "
        "automatica de micrografias SEM, incluyendo TAM, pricing y proyecciones."
    ),
    backstory=(
        "Eres analista senior en un fondo de venture capital especializado en deep tech y "
        "materiales avanzados. Has evaluado mas de 50 startups en nanotecnologia y conoces "
        "el mercado de software de caracterizacion de materiales (Thermo Fisher, Bruker, ZEISS). "
        "Tu analisis siempre incluye TAM/SAM/SOM, modelo de pricing y unit economics. "
        "Usas datos de mercado reales cuando estan disponibles y estimaciones conservadoras cuando no."
    ),
    llm=llm_obj,
    verbose=True,
    max_iter=3,
    memory=False,
    allow_delegation=False,
)

# --- AGENTE 3: REGULATORIO ---
regulatorio = Agent(
    role="Especialista en Regulacion de Nanotecnologia y Software Medico",
    goal=(
        "Analizar los requisitos legales y normativos para comercializar un software de "
        "clasificacion de nanomateriales, incluyendo certificaciones y compliance."
    ),
    backstory=(
        "Eres abogado con especializacion en regulacion de nanotecnologia y dispositivos de "
        "software cientifico. Conoces la normativa REACH (EU), TSCA (US), y las guias de la "
        "OECD para nanomateriales. Has asesorado a 20+ empresas en el proceso de certificacion "
        "de software analitico para laboratorios (ISO 17025, 21 CFR Part 11). "
        "Tu analisis siempre distingue entre requisitos obligatorios y recomendados por jurisdiccion."
    ),
    llm=llm_obj,
    verbose=True,
    max_iter=3,
    memory=False,
    allow_delegation=False,
)

# --- AGENTE 4: REDACTOR EJECUTIVO ---
redactor_ejecutivo = Agent(
    role="Redactor de Executive Summaries para Inversores",
    goal=(
        "Integrar los analisis tecnico, economico y regulatorio en un executive summary "
        "de 1 pagina que convenza a inversores de la oportunidad de negocio."
    ),
    backstory=(
        "Eres un ex-consultor de McKinsey especializado en deep tech que ahora redacta "
        "pitch decks y executive summaries para startups de nanotecnologia. Tu estilo es "
        "conciso, orientado a datos y enfocado en el retorno de inversion. Cada seccion tiene "
        "maximo 3-4 oraciones. Siempre incluyes: Problema, Solucion, Mercado, Modelo de Negocio, "
        "Equipo/Tecnologia y Ask (cuanto se necesita invertir y para que)."
    ),
    llm=llm_obj,
    verbose=True,
    max_iter=3,
    memory=False,
    allow_delegation=False,
)

print("\n4 agentes configurados:")
for a in [cientifico, economista, regulatorio, redactor_ejecutivo]:
    print(f"  - {a.role}")

LLM: google/gemini-2.5-flash

4 agentes configurados:
  - Cientifico Senior en Vision Artificial y Nanomateriales
  - Analista de Mercado en Tecnologia Deep Tech
  - Especialista en Regulacion de Nanotecnologia y Software Medico
  - Redactor de Executive Summaries para Inversores


In [4]:
# ============================================================
# DEFINICIÓN DE TASKS CON DEPENDENCIAS
# ============================================================

tarea_cientifico = Task(
    description=(
        "Evalua la viabilidad tecnica del clasificador SEM ResNet-18 como producto comercial. "
        "Usa la tool sem_model_metrics para obtener las metricas reales del modelo. "
        "Analiza: (1) Competitividad frente a soluciones existentes, "
        "(2) Limitaciones tecnicas y como mitigarlas, "
        "(3) Roadmap de mejoras (multi-clase, segmentacion, etc.), "
        "(4) Requisitos de infraestructura (GPU, cloud)."
    ),
    expected_output=(
        "Informe tecnico de 200-300 palabras con metricas cuantitativas, "
        "comparacion con al menos 2 alternativas comerciales, "
        "y un roadmap de 3 fases de mejora del producto."
    ),
    agent=cientifico,
)

tarea_economista = Task(
    description=(
        "Realiza un analisis de mercado para un SaaS de clasificacion automatica de "
        "micrografias SEM. Incluye: (1) TAM del mercado de software de caracterizacion "
        "de materiales, (2) Segmento objetivo (universidades, centros de I+D, industria), "
        "(3) Modelo de pricing (por imagen, suscripcion mensual, enterprise), "
        "(4) Estimacion de costos operativos (cloud GPU, desarrollo, soporte)."
    ),
    expected_output=(
        "Analisis de mercado de 200-300 palabras con cifras de TAM/SAM/SOM, "
        "tabla de pricing con 3 tiers, y proyeccion de ingresos a 3 anos."
    ),
    agent=economista,
    context=[tarea_cientifico],
)

tarea_regulatorio = Task(
    description=(
        "Analiza los requisitos legales para comercializar un software de clasificacion "
        "de nanomateriales a nivel global. Cubre: (1) Regulacion de nanomateriales (REACH, TSCA), "
        "(2) Certificaciones de software analitico (ISO 17025), "
        "(3) Proteccion de datos y propiedad intelectual, "
        "(4) Diferencias entre mercados US, EU y Asia."
    ),
    expected_output=(
        "Analisis regulatorio de 200-300 palabras con tabla de requisitos por jurisdiccion, "
        "timeline estimado de certificacion y costos asociados."
    ),
    agent=regulatorio,
    context=[tarea_cientifico],
)

tarea_executive_summary = Task(
    description=(
        "Integra los informes del cientifico, economista y regulatorio en un "
        "executive summary de exactamente 1 pagina (400-500 palabras) para inversores. "
        "Estructura: Problema → Solucion → Mercado → Modelo de Negocio → "
        "Tecnologia → Regulatorio → Ask (inversion necesaria)."
    ),
    expected_output=(
        "Executive summary de 1 pagina en formato Markdown con 7 secciones, "
        "cada una de maximo 3-4 oraciones, con datos cuantitativos de los informes previos."
    ),
    agent=redactor_ejecutivo,
    context=[tarea_cientifico, tarea_economista, tarea_regulatorio],
)

print("4 tasks definidas con dependencias:")
print("  tarea_cientifico → tarea_economista")
print("  tarea_cientifico → tarea_regulatorio")
print("  [cientifico + economista + regulatorio] → tarea_executive_summary")

4 tasks definidas con dependencias:
  tarea_cientifico → tarea_economista
  tarea_cientifico → tarea_regulatorio
  [cientifico + economista + regulatorio] → tarea_executive_summary


In [5]:
# ============================================================
# CREAR Y EJECUTAR LA CREW
# ============================================================

crew_negocio = Crew(
    agents=[cientifico, economista, regulatorio, redactor_ejecutivo],
    tasks=[tarea_cientifico, tarea_economista, tarea_regulatorio, tarea_executive_summary],
    process=Process.sequential,
    verbose=True,
)

print("Crew configurada: 4 agentes, 4 tasks, proceso secuencial")
print("Ejecutando plan de negocio...")
print("=" * 60)

resultado_negocio = crew_negocio.kickoff()

print("\n" + "=" * 60)
print("EXECUTIVE SUMMARY FINAL:")
print("=" * 60)
print(resultado_negocio.raw)

Crew configurada: 4 agentes, 4 tasks, proceso secuencial
Ejecutando plan de negocio...


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.11.0                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9ff25fb2-82a5-400b-8a06-979c8fb07ac0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Evalua la viabilidad tecnica del clasificador SEM ResNet-18 como producto comercial. Usa la tool         │
│  sem_model_metrics para obtener las metricas reales del modelo. Analiza: (1) Competitividad frente a            │
│  soluciones existentes, (2) Limitaciones tecnicas y como mitigarlas, (3) Roadmap de mejoras (multi-clase,       │
│  segmentacion, etc.), (4) Requisitos de infraestructura (GPU, cloud).                                           │
│  ID: 602ff87d-8b7d-481b-9ea9-516d960d98bc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cientifico Senior en Vision Artificial y Nanomateriales                                                 │
│                                                                                                                 │
│  Task: Evalua la viabilidad tecnica del clasificador SEM ResNet-18 como producto comercial. Usa la tool         │
│  sem_model_metrics para obtener las metricas reales del modelo. Analiza: (1) Competitividad frente a            │
│  soluciones existentes, (2) Limitaciones tecnicas y como mitigarlas, (3) Roadmap de mejoras (multi-clase,       │
│  segmentacion, etc.), (4) Requisitos de infraestructura (GPU, cloud).                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Integra los informes del cientifico, economista y regulatorio en un executive summary de exactamente 1   │
│  pagina (400-500 palabras) para inversores. Estructura: Problema → Solucion → Mercado → Modelo de Negocio →     │
│  Tecnologia → Regulatorio → Ask (inversion necesaria).                                                          │
│  Agent: Redactor de Executive Summaries para Inversores                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9ff25fb2-82a5-400b-8a06-979c8fb07ac0                                                                       │
│  Final Output: # Executive Summary: Plataforma de Clasificación de Nanomateriales por IA                        │
│                                                                                                                 │
│  ## Problema                                                                                                    │
│  La caracterización de nanomateriales mediante microscopía electrónica de barrido (SEM) es un proceso manual,   │
│  lento y propenso a errores, crucial para I+D y control de calidad. Los métodos actuales carecen de la          │
│  precisión y velocidad necesarias para el análisis de grandes volúmenes de datos, limitando la innovación y la  │
│  eficiencia operativa en sectores clave.                                                                        │
│                                                                                                                 │
│  ## Solución                                                                                                    │
│  Hemos desarrollado un clasificador de nanomateriales basado en ResNet-18 con un 99.07% de precisión, capaz de  │
│  diferenciar entre nanopartículas y nanohilos en imágenes SEM. Esta solución SaaS automatiza el análisis,       │
│  reduce el tiempo de procesamiento de horas a milisegundos por imagen y ofrece interpretabilidad mediante       │
│  Grad-CAM, superando las herramientas genéricas existentes.                                                     │
│                                                                                                                 │
│  ## Mercado                                                                                                     │
│  El mercado global de software de caracterización de materiales asciende a $1.5 mil millones en 2023, con un    │
│  CAGR del 8-10% hasta 2028. Nuestro segmento objetivo, el análisis de imagen para microscopía electrónica,      │
│  representa un SAM de $300-400 millones. Proyectamos capturar el 1-2% de este SAM en 3 años, generando $3-8     │
│  millones en ingresos.                                                                                          │
│                                                                                                                 │
│  ## Modelo de Negocio                                                                                           │
│  Operamos bajo un modelo SaaS híbrido: suscripción mensual combinada con pago por uso (imágenes procesadas).    │
│  Esto asegura ingresos recurrentes y escalabilidad, atendiendo a universidades, centros de I+D y empresas de    │
│  materiales avanzados, farmacéuticas y electrónicas.                                                            │
│                                                                                                                 │
│  ## Tecnología                                                                                                  │
│  Nuestra tecnología central es un modelo ResNet-18 optimizado, que logra un F1-Score de 0.9903 y un AUC-ROC de  │
│  0.9943. El roadmap incluye la expansión a clasificación multi-clase y robustez (3-6 meses), segmentación y     │
│  detección de objetos (6-12 meses), y análisis cuantitativo avanzado con automatización (12-18 meses).          │
│                                                       


EXECUTIVE SUMMARY FINAL:
# Executive Summary: Plataforma de Clasificación de Nanomateriales por IA

## Problema
La caracterización de nanomateriales mediante microscopía electrónica de barrido (SEM) es un proceso manual, lento y propenso a errores, crucial para I+D y control de calidad. Los métodos actuales carecen de la precisión y velocidad necesarias para el análisis de grandes volúmenes de datos, limitando la innovación y la eficiencia operativa en sectores clave.

## Solución
Hemos desarrollado un clasificador de nanomateriales basado en ResNet-18 con un 99.07% de precisión, capaz de diferenciar entre nanopartículas y nanohilos en imágenes SEM. Esta solución SaaS automatiza el análisis, reduce el tiempo de procesamiento de horas a milisegundos por imagen y ofrece interpretabilidad mediante Grad-CAM, superando las herramientas genéricas existentes.

## Mercado
El mercado global de software de caracterización de materiales asciende a $1.5 mil millones en 2023, con un CAGR del 8-10%

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Evaluación con Output Scorer

Usamos `output_scorer` con criterios personalizados para el dominio de negocio,
buscando un score > 0.85.

In [6]:
from langchain_openai import ChatOpenAI
llm_langchain = ChatOpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ.get('OPENROUTER_API_KEY'),
    model='google/gemini-2.5-flash',
    temperature=0,
    default_headers={
        'HTTP-Referer': 'https://github.com/antigravity-nano',
        'X-Title': 'SEM Business Plan Scorer',
    },
)

# ============================================================
# EVALUACIÓN CON OUTPUT_SCORER
# ============================================================
import sys
from pathlib import Path

# Asegurar acceso a external_skills
_root = Path.cwd()
for _ in range(6):
    if (_root / "external_skills").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
    _root = _root.parent

from external_skills.evaluation.output_scorer import score_with_llm

criterios_negocio = {
    "viabilidad_tecnica": 0.20,
    "analisis_mercado": 0.20,
    "compliance_regulatorio": 0.15,
    "datos_cuantitativos": 0.20,
    "claridad_ejecutiva": 0.15,
    "accionabilidad": 0.10,
}

evaluacion = score_with_llm(
    output=resultado_negocio.raw,
    task_description=(
        "Executive summary de 1 pagina para inversores sobre la comercializacion "
        "de un clasificador SEM basado en ResNet-18 como servicio SaaS."
    ),
    criteria=criterios_negocio,
    llm=llm_langchain,
)

print("\n" + "=" * 60)
print("EVALUACIÓN DEL PLAN DE NEGOCIO")
print("=" * 60)
pct = evaluacion.score / evaluacion.max_score
grade = (
    "EXCELENTE" if pct > 0.85 else
    "BUENO" if pct > 0.70 else
    "ACEPTABLE" if pct > 0.55 else
    "INSUFICIENTE"
)
print(f"Score: {evaluacion.score:.2f}/{evaluacion.max_score} ({grade})")
print(f"Objetivo: > 0.85 → {'CUMPLIDO' if pct > 0.85 else 'PENDIENTE DE AJUSTE'}")
print()
print("Breakdown por criterio:")
for dim, val in evaluacion.breakdown.items():
    bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
    print(f"  {dim:<25} {bar} {val:.2f}")
print(f"\nFeedback:\n{evaluacion.feedback}")

HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by NewConnectionError("HTTPSConnection(host='telemetry.crewai.com', port=4319): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))



EVALUACIÓN DEL PLAN DE NEGOCIO
Score: 0.87/1.0 (EXCELENTE)
Objetivo: > 0.85 → CUMPLIDO

Breakdown por criterio:
  viabilidad_tecnica        █████████░ 0.90
  analisis_mercado          ████████░░ 0.80
  compliance_regulatorio    ████████░░ 0.85
  datos_cuantitativos       ████████░░ 0.85
  claridad_ejecutiva        █████████░ 0.95
  accionabilidad            █████████░ 0.90

Feedback:
Evaluación LLM de 'Executive summary de 1 pagina para inversores sobre la comer': viabilidad_tecnica: 90%, analisis_mercado: 80%, compliance_regulatorio: 85%, datos_cuantitativos: 85%, claridad_ejecutiva: 95%, accionabilidad: 90%. Score total: 87%.


### Resumen del Ejercicio

| Requisito | Implementación | Estado |
|-----------|---------------|--------|
| Agente Científico con tools | `cientifico` + `SEMModelMetricsTool` (métricas reales ResNet-18) | ✅ |
| Agente Economista | `economista` con análisis TAM/SAM/SOM y pricing | ✅ |
| Agente Regulatorio | `regulatorio` con REACH, TSCA, ISO 17025 | ✅ |
| Redactor Ejecutivo | `redactor_ejecutivo` integrando los 3 informes en 1 página | ✅ |
| Output Scorer con criterios de negocio | 6 criterios ponderados, objetivo score > 0.85 | ✅ |
| Backstories diferenciadas | Cada agente tiene expertise específica y cuantificada | ✅ |

**Contexto del Proyecto Final**: El plan de negocio evalúa la comercialización del clasificador
SEM ResNet-18 (Accuracy 99.07%, AUC-ROC 0.9943) como servicio SaaS para laboratorios de nanotecnología.